# TTS Bark playground

This notebook wil contain all the experiments and documentations regarding the [Piper](https://github.com/OHF-Voice/piper1-gpl) model.

>NOTE: the notebook has been executed on a machine with the following specs:

In [3]:
!winfetch


 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
                                  
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll

Fazei@LAPTOP-O9V24FCP
---------------------
OS: Windows 11 Home [64-bit]
Host: LENOVO 82K2
Kernel: 10.0.26200.0
Motherboard: LENOVO LNVNB161216
Uptime: 3 hours 35 minutes
Packages: 26 (choco)
Shell: PowerShell v5.1.26100.7920
Resolution: 1536x864
Terminal: Visual Studio Code
CPU: AMD Ryzen 5 5600H with Radeon Graphics @ 3.294GHz
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
GPU: AMD Radeon(TM) Graphics
Memory: 8,69 GiB / 13,86 GiB (62%)
Disk (C:): 442 GiB / 475 GiB (93%)

  

## Setting up a venv (Virtual Environment)

> this notebook uses Python 3.10.0

Before we start, make sure to have a venv (virtual environment) to run this model

this can be achieved in the following ways:

- Using python which will include pip
    ```
    python -m venv .venv-name
    ```

- Using UV which is a package manager for Python written in Rust
    ```
    uv venv .venv-name
    ```

## Python & Packages

To get TTS Bark working, make sure that your virtual environment has installed all the requirements in [requirements.txt](./requirements.txt)

this can be achieved in the following ways:

- Using pip (package manager for Python)
    ```
    pip install -r requirements
    ```

- Using uv (package manager for Python written in Rust)
    ```
    uv pip install -r requirements.txt
    ```

# Code implementation


To download voices see the following [documentation](https://rhasspy.github.io/piper-samples/) to see all the options and the [Python API usage](https://github.com/OHF-Voice/piper1-gpl/blob/main/docs/API_PYTHON.md) for the command to download these model(s).

for this example we have downloaded the voices ```en_US-libritts-high``` and ```en_US-norman-medium```

In [4]:
from piper import PiperVoice, SynthesisConfig
from tkinter.filedialog import askopenfilename
import re
import tkinter
import wave
import time

"""
Simple implementation of PiperTTS in a class
"""
class PiperTTS:

    """
    All paths regarding input and output
    """
    # Maybe make it configurable if there are multiple models
    path_voice = "./weights/en_US-libritts-high.onnx"
    path_config = "./weights/en_US-libritts-high.onnx.json"
    path_output = "../outputs/piper-output.wav"
    
    """
    Main functions that first initialized the input and then generatees the TTS
    """
    def __init__(self):
        input = self.initialize_input()
        self.generate_tts(input)

    """
    Parent function of input cycle
    """
    def initialize_input(self):
        input_path = self.retrieve_input_path()
        return self.process_input(input_path)
        
    """
    Retrieves input file, should only allow .txt for now.
    It opens the file explorer in the current directory
    Also hides the TK window
    """
    def retrieve_input_path(self):
        print("Please provide the text that you would like to process")

        tkinter.Tk().withdraw()
        return askopenfilename(
            initialdir="./",
            title="Select input file", 
            filetypes=[("Text Files", "*.txt")]
            )
    
    """
    Uses the retrieved path and cleans it up with regex
    Also removed unnecassary double \n
    """
    def process_input(self, path: str):
        print(f"Processing text from {path}")

        with open(path, "r") as f:
            # funky magic, maybe put it into multiple steps for readability
            return re.sub(r"[\"']", 
                          "", 
                          f.read()).replace("\n\n", "\n")

    """
    Generates the tts with the clean input
    Has a config that can be tweaked
    also checks if the file has been initialized based on the first chunk
    afterwards it saves the output in the general outputs folder
    """
    def generate_tts(self, input):
        print("Generating output...")

        syn_config = SynthesisConfig(
            volume=0.8,
            length_scale=0.9,
            noise_scale=0.667,    
            noise_w_scale=0.8,
            normalize_audio=False   
        )

        piper: PiperVoice = PiperVoice.load(self.path_voice, config_path=self.path_config) 

        piper.synthesize(..., syn_config=syn_config)

        durations = []

        with wave.open(self.path_output, "wb") as output_file:
            init_output_file = False
            i = 0

            for chunk in piper.synthesize(input):
                if not init_output_file:
                    output_file.setnchannels(1)
                    output_file.setsampwidth(2)
                    output_file.setframerate(chunk.sample_rate)
                    init_output_file=True

                print(f"{i} | Generating chunk: {chunk.phonemes}")
                ta = time.perf_counter()
                output_file.writeframes(chunk.audio_int16_bytes)
                te = time.perf_counter()
                print(f"Generated chunk in {te - ta:.4f} seconds")
                i += 1

                duration =  te - ta
                durations.append(duration)

        if durations:
            avg_time = sum(durations) / len(durations)
            print(f"\n--- Statistics ---")
            print(f"Total chunks: {len(durations)}")
            print(f"Average time per chunk: {avg_time:.4f} seconds")
            print(f"Total processing time: {sum(durations):.4f} seconds")

        return print(f"Successfully generated audio at {self.path_output}")
    
PiperTTS()


Please provide the text that you would like to process
Processing text from C:/Users/Fazei/HvA-HBO-ICT/repos/SociRo/playground/tts/tts.txt
Generating output...
0 | Generating chunk: ['ð', 'ə', ' ', 'd', 'ɹ', 'ˈ', 'æ', 'ɡ', 'ə', 'n', ' ', 'ɪ', 'n', 'ð', 'ə', ' ', 'n', 'ˈ', 'i', 'ː', 'd', 'ə', 'l', ' ', 'm', 'ˈ', 'i', 'ə', ' ', 'w', 'ʌ', 'z', ' ', 't', 'ˈ', 'ɛ', 'n', ' ', 'j', 'ˈ', 'ɪ', 'ɹ', 'z', ' ', 'ˈ', 'o', 'ʊ', 'l', 'd', ',', ' ', 'æ', 'n', 'd', ' ', 'ʃ', 'i', 'ː', ' ', 'h', 'ɐ', 'd', 'b', 'ɪ', 'n', ' ', 'ɪ', 'n', 'ð', 'ə', ' ', 'h', 'ˈ', 'ɑ', 'ː', 's', 'p', 'ɪ', 'ɾ', 'ə', 'l', ' ', 'f', 'ɔ', 'ː', 'ɹ', ' ', 't', 'ˈ', 'u', 'ː', ' ', 'd', 'ˈ', 'e', 'ɪ', 'z', '.']
Generated chunk in 0.0004 seconds
1 | Generating chunk: ['ʃ', 'i', 'ː', ' ', 'd', 'ˈ', 'ɪ', 'd', 'n', 't', ' ', 'm', 'ˈ', 'a', 'ɪ', 'n', 'd', ' ', 'ð', 'ə', ' ', 'b', 'ˈ', 'ɛ', 'd', ',', ' ', 'ɔ', 'ː', 'ɹ', ' ', 'ˈ', 'i', 'ː', 'v', 'ə', 'n', ' ', 'ð', 'ə', ' ', 'f', 'ˈ', 'u', 'ː', 'd', '.']
Generated chunk in 0.0005 seconds
2

Missing phoneme from id map: ̩


61 | Generating chunk: ['j', 'u', 'ː', ' ', 'd', 'ˈ', 'ɪ', 'd', ' ', 'b', 'ɹ', 'ˈ', 'ɪ', 'l', 'i', 'ə', 'n', 't', 'l', 'i', '.']
Generated chunk in 0.0004 seconds
62 | Generating chunk: ['m', 'ˈ', 'i', 'ə', ' ', 'p', 'ɹ', 'ˈ', 'ɛ', 's', 't', ' ', 'ð', 'ə', ' ', 'k', 'ˈ', 'ɑ', 'ː', 'ʔ', 'n', '̩', ' ', 'p', 'ˈ', 'æ', 'd', ' ', 't', 'ə', ' ', 'h', 'ɜ', 'ː', 'ɹ', ' ', 'ˈ', 'ɑ', 'ː', 'ɹ', 'm', ' ', 'æ', 'n', 'd', ' ', 'f', 'ˈ', 'ɛ', 'l', 't', ' ', 's', 't', 'ɹ', 'ˈ', 'e', 'ɪ', 'n', 'd', 'ʒ', ' ', 'æ', 'n', 'd', ' ', 'p', 'ɹ', 'ˈ', 'a', 'ʊ', 'd', ' ', 'æ', 'n', 'd', ' ', 'ɐ', ' ', 'l', 'ˈ', 'ɪ', 'ɾ', 'ə', 'l', ' ', 'w', 'ˈ', 'ɑ', 'ː', 'b', 'l', 'i', ',', ' ', 'ˈ', 'ɔ', 'ː', 'l', ' ', 'ɐ', 't', 'w', 'ˈ', 'ʌ', 'n', 's', '.']
Generated chunk in 0.0003 seconds
63 | Generating chunk: ['ʃ', 'i', 'ː', ' ', 'θ', 'ˈ', 'ɔ', 'ː', 't', ' ', 'ʌ', 'v', ' ', 's', 'ˈ', 'ɛ', 'ɹ', 'ə', 'n', ' ', 'w', 'ˈ', 'ɔ', 'ː', 'k', 'ɪ', 'ŋ', ' ', 'd', 'ˌ', 'a', 'ʊ', 'n', ' ', 'ð', 'ə', ' ', 'm', 'ˈ', 'a', 'ʊ', 'n', 't', 

# Findings

- the ```en_US-norman-medium``` seems to hallucinate at the 10-12th chunks, it also sounded really robotic

- the ```en_US-libritts-high``` sounds natural, but it is like listening to a taecher presenting a scientific report

- the speed at which it generates the chunks is impressive, piper specializes in running on lower end hard ware

- We might be able to run this model on the raspberry pi itself, although I would prefer a server client architecture